In [21]:
import torch
from transformers import BertTokenizer, BertModel

# Load BERT model for embeddings
bert_model = BertModel.from_pretrained("bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Input word
entity_word = "Apple"

# Desired embedding dimension
desired_dimension = 300  # Change this to your desired dimension

# Step 1: Tokenization
tokens = tokenizer.tokenize(entity_word)
input_ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(input_ids).unsqueeze(0)  # Batch dimension added

# Step 2: Get BERT embeddings for the entity word
with torch.no_grad():
    outputs = bert_model(input_ids)

# Word-level embeddings for the entity word with desired dimension
entity_word_embedding = outputs.last_hidden_state[:, 0, :desired_dimension]


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [22]:
import embedding

In [23]:
import pickle
import torch
with open("D:\\Entity Aspect Linking\\data\picklefiles\\final_eal.pkl", 'rb') as eal:
    data = pickle.load(eal)

ent = [data[i][0] for i in range(len(data))]

asp = [data[i][1] for i in range(len(data))]

In [24]:
ent[0]

{'id': '0',
 'target_entity': 'Gautama Buddha',
 'paragraph': "Srivastava's discovery of the terracotta sealings bearing the name Kapilavastu has led some scholars to believe that modern-day Piprahwa was the site of the ancient city of Kapilavastu, the capital of the Shakya kingdom, where Siddhartha Gautama spent the first 29 years of his life. Others suggest that the original site of Kapilavastu is located 16 km to the northwest, at Tilaurakot, in what is currently Kapilvastu District in Nepal. This question is especially important to scholars of Buddhist history, as Kapilavastu was the capital of the Shakya kingdom. King Śuddhodana and Queen Māyādevī lived at Kapilavastu, as did their son Prince Siddhartha Gautama until he left the palace at 29 years of age.",
 'entities': [{'eid': '00',
   'entity': 'Kapilavastu (ancient city)',
   'mention': 'Kapilavastu'},
  {'eid': '01', 'entity': 'Shakya', 'mention': 'Shakya'},
  {'eid': '02', 'entity': 'Gautama Buddha', 'mention': 'Siddhartha G

In [25]:
target_emb = torch.zeros(len(ent), 768)
pretrained = 'bert-base-uncased'
ent_emb = embedding.EntityEmbedding(pretrained = pretrained)
for i in range(len(ent)):
    entity_word = ent[i]['target_entity']
    tokens = tokenizer.tokenize(entity_word)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0) 
    target_emb[i] = ent_emb(input_ids)
    
    

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [26]:
count = 0
for item in ent:
    count += len(item['entities'])
    
count

467

In [27]:
t_ent_emb = torch.zeros(count, 768)

for i in range(len(ent)):
    for el in ent[i]['entities']:
        word = el['entity']
        tokens = tokenizer.tokenize(word)
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        input_ids = torch.tensor(input_ids).unsqueeze(0)
        t_ent_emb[i] = ent_emb(input_ids)

In [28]:
asp_emb = torch.zeros(len(asp), 768)
for i in range(len(asp)):
    aspect = asp[i]['true_aspect']
    tokens = tokenizer.tokenize(aspect)
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0) 
    asp_emb[i] = ent_emb(input_ids)

In [32]:
asp[0]

{'id': '0',
 'true_aspect': 'Biography',
 'candidate_aspects': [{'id': 'A00',
   'aspect_name': 'Historical Siddhārtha Gautama',
   'section_heading': ['Historical Siddhārtha Gautama'],
   'content': 'Scholars are hesitant to make unqualified claims about the historical facts of the Buddha\'s life. Most people accept that the Buddha lived, taught, and founded a monastic order during the Mahajanapada era during the reign of Bimbisara (, or c. 400 BCE), the ruler of the Magadha empire, and died during the early years of the reign of Ajatasatru, who was the successor of Bimbisara, thus making him a younger contemporary of Mahavira, the Jain tirthankara. While the general sequence of "birth, maturity, renunciation, search, awakening and liberation, teaching, death" is widely accepted, there is less consensus on the veracity of many details contained in traditional biographies.\nThe times of Gautama\'s birth and death are uncertain. Most historians in the early 20th century dated his lifeti

In [31]:
asp_count = 0
for item in asp:
    for el in item['candidate_aspects']:
        asp_count += len(el['entities'])
asp_count

18252

In [34]:
a_ent_emb = torch.zeros(asp_count, 768)
for i in range(len(asp)):
    for el in asp[i]['candidate_aspects']:
        for ent in el['entities']:
            word = ent['entity_name']
            tokens = tokenizer.tokenize(word)
            input_ids = tokenizer.convert_tokens_to_ids(tokens)
            input_ids = torch.tensor(input_ids).unsqueeze(0) 
            a_ent_emb[i] = ent_emb(input_ids)
            